# 03 · Resident-only prototype (เงื่อนไข C) เทียบกับ A และ B (ระยะที่ 4, RQ1)

ทุกวัน: agent อ่านรายงานผู้ป่วยของเมื่อวาน + สุขภาพตนเอง → LLM เลือก action → รวมสัดส่วนแบบถ่วงน้ำหนัก → $q_{out}$ → $\beta_{\mathrm{eff}}$ → SEIR 1 วัน → บันทึก → checkpoint

- ถ้า Colab หลุด: รัน notebook ใหม่ตั้งแต่ต้น การรันจะ **ต่อจากวันล่าสุด** ที่บันทึกไว้ (run id เดิม)
- run ที่เสร็จแล้วจะถูกข้าม
- ตั้ง `BACKEND = "mock"` เพื่อทดสอบ pipeline โดยไม่ใช้ GPU (ผลไม่ใช่ผลการทดลอง)

In [ ]:
# ---- Setup (same cell in every notebook) ----
import os, sys, subprocess
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/ThaiGABM")   # where you uploaded the abm folder
    IN_COLAB = True
except ImportError:                                   # running locally (notebooks 00 and 01)
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    IN_COLAB = False

assert (ROOT / "gabm").exists(), f"gabm/ not found in {ROOT}: upload the abm folder to MyDrive/ThaiGABM"
sys.path.insert(0, str(ROOT))
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements-colab.txt")], check=True)

from gabm import load_config
cfg = load_config(ROOT / "config" / "base.yaml")
FIG = ROOT / "outputs" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
print("ROOT:", ROOT, "| Colab:", IN_COLAB)
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True,
                     capture_output=True, text=True).stdout or "no GPU")

In [ ]:
# Hugging Face token: needed only for gated models (llama8b). Add it in Colab: key icon (Secrets) -> HF_TOKEN
if IN_COLAB:
    from google.colab import userdata
    try:
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
        print("HF_TOKEN loaded")
    except Exception:
        print("No HF_TOKEN secret: gated models will fail to download")

In [ ]:
MODEL_KEY = "qwen7b"              # @param ["typhoon8b", "qwen7b", "llama8b", "openthaigpt7b", "seallm7b", "mistral7b", "qwen1_5b"]
BACKEND = cfg["llm"]["backend"]   # "mock" to test without a GPU
SEEDS = cfg["sim"]["seeds"]
DAYS = cfg["sim"]["days"]         # pilot horizon (base.yaml = 30); set 180 here for a full-epidemic run
cfg["sim"]["days"] = DAYS         # applies to A/B/C below so all curves share the same horizon
RUN_REPRO_CHECK = False           # two extra temperature-0 runs to check reproducibility

In [ ]:
import copy
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from gabm.sim import run_simulation, contact_model_from_cfg
from gabm.llm import load_backend
from gabm.analysis import load_runs, load_daily, plot_epidemic, plot_action_shares

personas = pd.read_csv(ROOT / cfg["personas"]["file"])
cm = contact_model_from_cfg(cfg, ROOT)
n, D = len(personas), cfg["sim"]["days"]
print(f"Budget: G = {n} agents x {D} days x 1 model x {len(SEEDS)} seeds = {n * D * len(SEEDS):,} resident generations (+ retries)")

base = [run_simulation(cfg, c, seed=0, cm=cm, root=ROOT, verbose=False) for c in "AB"]   # cheap, CPU

In [ ]:
backend = load_backend(cfg["llm"], MODEL_KEY, BACKEND)
runs_C = [run_simulation(cfg, "C", seed=s, personas=personas, backend=backend, model_key=MODEL_KEY, cm=cm, root=ROOT)
          for s in SEEDS]

## ผลลัพธ์

In [ ]:
ids = [s.run_id for s in base + runs_C]
daily = load_daily(ids, ROOT)
summary = load_runs(ROOT)
summary = summary[summary.run_id.isin(ids)]
display(summary.drop(columns=["gpu"]).round(4))

fig, axes = plt.subplots(2, 2, figsize=(13, 7))
plot_epidemic(daily, ax=axes[0, 0], y="I"); axes[0, 0].set_title("Infectious I")
plot_epidemic(daily, ax=axes[0, 1], y="q_out"); axes[0, 1].set_title("q_out (out-of-home contact vs normal day)")
plot_action_shares(daily[daily.condition == "C"], ax=axes[1, 0], title=f"Condition C action shares ({MODEL_KEY}, mean of seeds)")
plot_epidemic(daily, ax=axes[1, 1], y="reported_per100k"); axes[1, 1].set_title("Reported /100k/day")
plt.tight_layout(); plt.savefig(FIG / f"03_resident_{MODEL_KEY}.png", dpi=150); plt.show()

In [ ]:
# Behaviour by group and by own symptoms (from the decision log)
import json
dec = pd.concat([pd.read_json(ROOT / "outputs" / "runs" / s.run_id / "decisions.jsonl", lines=True).assign(seed=s.run_id) for s in runs_C], ignore_index=True)
dec["sick"] = dec.health.str.contains("fever")
display(pd.crosstab(dec.group, dec.choice, normalize="index").round(2))
display(pd.crosstab(dec.sick, dec.choice, normalize="index").round(2))
print(f"retried {dec.retried.mean():.1%}, fallback {dec.fallback.mean():.1%} of {len(dec):,} decisions")

rt = summary[summary.condition == "C"][["run_id", "runtime_s", "n_decisions", "valid_first_try", "fallback_rate", "gpu"]]
rt.assign(runtime_min=rt.runtime_s / 60).round(3)

## ตรวจการทำซ้ำได้ (หัวข้อ 12)
config เดิม + temperature 0 ควรให้ `daily.csv` เหมือนกัน (บน GPU ชนิดเดียวกัน)

In [ ]:
if RUN_REPRO_CHECK:
    c0 = copy.deepcopy(cfg); c0["llm"]["temperature"] = 0.0
    a, b = [run_simulation(c0, "C", seed=SEEDS[0], personas=personas, backend=backend, model_key=MODEL_KEY,
                           cm=cm, root=ROOT, tag=f"repro_{t}") for t in "ab"]
    da, db = [pd.read_csv(s.path("daily.csv")).drop(columns=["seconds", "llm_seconds"]) for s in (a, b)]
    print("identical daily.csv:", da.round(8).equals(db.round(8)))